# Notebook 9: Machine Learning Evaluation and Comparison

## Objective

This notebook conducts the final out-of-sample evaluation of the forecasting models developed in Notebook 08.

The analysis uses the locked January–December 2025 test period to:

1. evaluate the persistence, Ridge, Random Forest, and XGBoost forecasts
2. compare symmetric and asymmetric exchange-rate representations
3. examine performance across food subclasses and months
4. identify where forecasting errors are concentrate
5. compare machine-learning forecasts with time-aligned econometric baselines

No models are tuned or selected using the test results.

In [64]:
# import libraries
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

sns.set_theme(style="whitegrid")

print("Evaluation libraries imported successfully.")

Evaluation libraries imported successfully.


In [65]:
# define input and output locations
processed_data_directory = Path("../data/processed")
ml_table_directory = Path("../reports/tables/machine_learning")
econometric_table_directory = Path(
    "../reports/tables/econometrics"
)

evaluation_table_directory = Path(
    "../reports/tables/model_evaluation"
)
evaluation_figure_directory = Path(
    "../reports/figures/model_evaluation"
)

evaluation_table_directory.mkdir(parents=True, exist_ok=True)
evaluation_figure_directory.mkdir(parents=True, exist_ok=True)


# load the locked outcomes and modelling outputs
ml_data = pd.read_csv(
    processed_data_directory / "ml_model_data.csv",
    parse_dates=["Date"],
)

ml_test_predictions = pd.read_csv(
    ml_table_directory / "ml_test_predictions.csv",
    parse_dates=["Date"],
)

validation_model_results = pd.read_csv(
    ml_table_directory / "validation_model_comparison.csv"
)

test_actuals = ml_data.loc[
    ml_data["Split"] == "Test",
    [
        "Date",
        "ClassDescription",
        "SubclassDescription",
        "Food_Inflation_Pct",
    ],
].copy()

print("Machine-learning data loaded:", len(ml_data))
print("Locked test outcomes loaded:", len(test_actuals))
print("Forecast rows loaded:", len(ml_test_predictions))
print("Validation models loaded:", len(validation_model_results))

Machine-learning data loaded: 4554
Locked test outcomes loaded: 552
Forecast rows loaded: 3864
Validation models loaded: 7


In [66]:
# validate the evaluation sample
identifier_columns = [
    "Date",
    "ClassDescription",
    "SubclassDescription",
]

prediction_identifier_columns = identifier_columns + [
    "Model",
    "Representation",
]

prediction_counts = (
    ml_test_predictions.groupby(
        ["Model", "Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Predictions"})
)

evaluation_checks = pd.DataFrame(
    {
        "Check": [
            "Test sample contains 552 outcomes",
            "Test sample covers 46 food subclasses",
            "Test sample covers 12 months",
            "Test outcomes contain no duplicate keys",
            "Predictions contain seven model variants",
            "Every model variant contains 552 predictions",
            "Predictions contain no duplicate records",
            "Target was absent from prediction file",
        ],
        "Passed": [
            len(test_actuals) == 552,
            test_actuals["SubclassDescription"].nunique() == 46,
            test_actuals["Date"].nunique() == 12,
            not test_actuals.duplicated(identifier_columns).any(),
            len(prediction_counts) == 7,
            prediction_counts["Predictions"].eq(552).all(),
            not ml_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
            "Food_Inflation_Pct"
            not in ml_test_predictions.columns,
        ],
    }
)

if not evaluation_checks["Passed"].all():
    failed_checks = evaluation_checks.loc[
        ~evaluation_checks["Passed"],
        "Check",
    ].tolist()
    raise ValueError(f"Evaluation checks failed: {failed_checks}")


# Attach actual outcomes for final evaluation
forecast_evaluation_data = ml_test_predictions.merge(
    test_actuals,
    on=identifier_columns,
    how="left",
    validate="many_to_one",
)

missing_actuals = int(
    forecast_evaluation_data["Food_Inflation_Pct"]
    .isna()
    .sum()
)

if missing_actuals:
    raise ValueError(
        f"{missing_actuals} predictions could not be matched "
        "to test outcomes."
    )

evaluation_sample_summary = pd.DataFrame(
    {
        "Value": [
            len(test_actuals),
            test_actuals["SubclassDescription"].nunique(),
            test_actuals["Date"].nunique(),
            test_actuals["Date"].min(),
            test_actuals["Date"].max(),
            len(prediction_counts),
            len(forecast_evaluation_data),
            missing_actuals,
        ]
    },
    index=[
        "Test outcomes",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Forecast variants",
        "Forecast-evaluation rows",
        "Missing matched outcomes",
    ],
)

display(evaluation_checks)
display(evaluation_sample_summary)
display(prediction_counts)

print(
    "All evaluation setup checks passed:",
    bool(evaluation_checks["Passed"].all()),
)

,Check,Passed
0,Test sample contains 552 outcomes,True
1,Test sample covers 46 food subclasses,True
2,Test sample covers 12 months,True
3,Test outcomes contain no duplicate keys,True
4,Predictions contain seven model variants,True
5,Every model variant contains 552 predictions,True
6,Predictions contain no duplicate records,True
7,Target was absent from prediction file,True


,Value
Test outcomes,552
Food subclasses,46
Unique months,12
Start date,2025-01-01 00:00:00
End date,2025-12-01 00:00:00
Forecast variants,7
Forecast-evaluation rows,3864
Missing matched outcomes,0


,Model,Representation,Predictions
0,Persistence,Lag-1 benchmark,552
1,Random Forest,Asymmetric,552
2,Random Forest,Symmetric,552
3,Ridge,Asymmetric,552
4,Ridge,Symmetric,552
5,XGBoost,Asymmetric,552
6,XGBoost,Symmetric,552


All evaluation setup checks passed: True


## Overall locked-test performance

The following evaluation compares all seven forecast variants across the complete 2025 test sample.

In addition to MAE, RMSE, R², and directional accuracy, mean bias is reported. A bias is calculated as predicted inflation minus observed inflation. A positive value therefore indicates average overprediction, while a negative value indicates average underprediction.

In [67]:
# calculate overall test metrics
def calculate_forecast_metrics(evaluation_data):
    actual_values = evaluation_data[
        "Food_Inflation_Pct"
    ].to_numpy()

    predicted_values = evaluation_data[
        "Predicted_Food_Inflation_Pct"
    ].to_numpy()

    forecast_errors = predicted_values - actual_values

    return {
        "Observations": len(evaluation_data),
        "MAE": mean_absolute_error(
            actual_values,
            predicted_values,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual_values,
                predicted_values,
            )
        ),
        "R2": r2_score(
            actual_values,
            predicted_values,
        ),
        "Directional_Accuracy_Pct": (
            np.mean(
                np.sign(actual_values)
                == np.sign(predicted_values)
            )
            * 100
        ),
        "Mean_Bias": np.mean(forecast_errors),
    }


test_metric_records = []

for (
    model_name,
    representation,
), model_data in forecast_evaluation_data.groupby(
    ["Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(model_data)

    test_metric_records.append(
        {
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

ml_test_model_results = pd.DataFrame(test_metric_records)

persistence_result = ml_test_model_results.loc[
    ml_test_model_results["Model"] == "Persistence"
].iloc[0]

ml_test_model_results[
    "RMSE_Improvement_vs_Persistence_Pct"
] = (
    (
        persistence_result["RMSE"]
        - ml_test_model_results["RMSE"]
    )
    / persistence_result["RMSE"]
    * 100
)

ml_test_model_results[
    "MAE_Improvement_vs_Persistence_Pct"
] = (
    (
        persistence_result["MAE"]
        - ml_test_model_results["MAE"]
    )
    / persistence_result["MAE"]
    * 100
)

ml_test_model_results["Test_Rank"] = (
    ml_test_model_results["RMSE"]
    .rank(method="min")
    .astype(int)
)

ml_test_model_results = ml_test_model_results.sort_values(
    ["Test_Rank", "MAE"]
).reset_index(drop=True)

display(
    ml_test_model_results[
        [
            "Test_Rank",
            "Model",
            "Representation",
            "Observations",
            "MAE",
            "RMSE",
            "R2",
            "Directional_Accuracy_Pct",
            "Mean_Bias",
            "RMSE_Improvement_vs_Persistence_Pct",
            "MAE_Improvement_vs_Persistence_Pct",
        ]
    ]
)

leading_test_model = ml_test_model_results.iloc[0]

print("Lowest test RMSE:")
print(
    f"{leading_test_model['Model']} — "
    f"{leading_test_model['Representation']}"
)
print(f"Test RMSE: {leading_test_model['RMSE']:.6f}")

,Test_Rank,Model,Representation,Observations,MAE,RMSE,R2,Directional_Accuracy_Pct,Mean_Bias,RMSE_Improvement_vs_Persistence_Pct,MAE_Improvement_vs_Persistence_Pct
0,1,XGBoost,Symmetric,552,1.060854,1.929075,0.113069,59.782609,0.263793,13.703894,20.423091
1,2,Random Forest,Symmetric,552,1.066689,1.964122,0.080550,63.405797,0.314879,12.136122,19.985359
2,3,XGBoost,Asymmetric,552,1.090656,1.973192,0.072038,61.231884,0.323877,11.730351,18.187517
3,4,Ridge,Symmetric,552,1.089515,1.984985,0.060913,60.869565,0.280788,11.202815,18.273135
4,5,Ridge,Asymmetric,552,1.092858,1.986590,0.059394,61.050725,0.287840,11.131018,18.022346
5,6,Random Forest,Asymmetric,552,1.075816,1.991030,0.055185,61.231884,0.313877,10.932393,19.300687
6,7,Persistence,Lag-1 benchmark,552,1.333117,2.235414,-0.190988,54.347826,0.025886,0.000000,0.000000


Lowest test RMSE:
XGBoost — Symmetric
Test RMSE: 1.929075


In [68]:
# examine the validation-to-test generalisation gap
validation_metrics = validation_model_results.rename(
    columns={
        "MAE": "Validation_MAE",
        "RMSE": "Validation_RMSE",
        "R2": "Validation_R2",
        "Directional_Accuracy_Pct": (
            "Validation_Directional_Accuracy_Pct"
        ),
    }
)

test_metrics = ml_test_model_results.rename(
    columns={
        "MAE": "Test_MAE",
        "RMSE": "Test_RMSE",
        "R2": "Test_R2",
        "Directional_Accuracy_Pct": (
            "Test_Directional_Accuracy_Pct"
        ),
    }
)

validation_test_comparison = validation_metrics.merge(
    test_metrics[
        [
            "Model",
            "Representation",
            "Test_MAE",
            "Test_RMSE",
            "Test_R2",
            "Test_Directional_Accuracy_Pct",
            "Mean_Bias",
            "Test_Rank",
        ]
    ],
    on=["Model", "Representation"],
    how="inner",
    validate="one_to_one",
)

validation_test_comparison[
    "RMSE_Generalisation_Gap"
] = (
    validation_test_comparison["Test_RMSE"]
    - validation_test_comparison["Validation_RMSE"]
)

validation_test_comparison[
    "RMSE_Change_Pct"
] = (
    validation_test_comparison["RMSE_Generalisation_Gap"]
    / validation_test_comparison["Validation_RMSE"]
    * 100
)

validation_test_comparison["Validation_Rank"] = (
    validation_test_comparison["Validation_RMSE"]
    .rank(method="min")
    .astype(int)
)

validation_test_comparison = validation_test_comparison.sort_values(
    "Test_Rank"
).reset_index(drop=True)

display(
    validation_test_comparison[
        [
            "Model",
            "Representation",
            "Validation_Rank",
            "Test_Rank",
            "Validation_RMSE",
            "Test_RMSE",
            "RMSE_Generalisation_Gap",
            "RMSE_Change_Pct",
            "Validation_R2",
            "Test_R2",
            "Validation_Directional_Accuracy_Pct",
            "Test_Directional_Accuracy_Pct",
        ]
    ]
)

,Model,Representation,Validation_Rank,Test_Rank,Validation_RMSE,Test_RMSE,RMSE_Generalisation_Gap,RMSE_Change_Pct,Validation_R2,Test_R2,Validation_Directional_Accuracy_Pct,Test_Directional_Accuracy_Pct
0,XGBoost,Symmetric,1,1,1.722070,1.929075,0.207006,12.020751,0.177449,0.113069,63.949275,59.782609
1,Random Forest,Symmetric,3,2,1.759446,1.964122,0.204676,11.632964,0.141356,0.080550,64.855072,63.405797
2,XGBoost,Asymmetric,2,3,1.728267,1.973192,0.244925,14.171711,0.171518,0.072038,63.405797,61.231884
3,Ridge,Symmetric,5,4,1.869568,1.984985,0.115417,6.173462,0.030509,0.060913,63.768116,60.869565
4,Ridge,Asymmetric,6,5,1.873179,1.986590,0.113410,6.054428,0.026759,0.059394,63.586957,61.050725
5,Random Forest,Asymmetric,4,6,1.796596,1.991030,0.194434,10.822330,0.104713,0.055185,63.949275,61.231884
6,Persistence,Lag-1 benchmark,7,7,2.340242,2.235414,-0.104828,-4.479351,-0.519088,-0.190988,56.340580,54.347826


### Interpretation of overall test performance

The validation-based selection was supported by the locked-test results. Symmetric XGBoost retained first place, with an RMSE of 1.929 and MAE of 1.061. It reduced RMSE by 13.7% and MAE by 20.4% relative to persistence.

All six machine-learning models outperformed persistence and produced positive test R² values. Persistence produced a negative R², indicating that it performed worse than predicting the test-sample mean.

Symmetric Random Forest ranked second by RMSE and achieved the highest
directional accuracy of 63.41%. Therefore, the model that minimised the size of forecast errors was not the model that most frequently predicted the correct sign of food inflation.

Test errors were higher than validation errors for all machine-learning models. For symmetric XGBoost, RMSE increased by approximately 12.0%, while R² declined from 0.177 to 0.113. This deterioration demonstrates the importance of retaining a genuinely unseen test period.

All machine-learning models had positive mean bias, indicating modest average overprediction during 2025. Nevertheless, they remained more accurate than the persistence benchmark.

The symmetric specification achieved a lower overall test RMSE than its
asymmetric counterpart for Ridge, Random Forest, and XGBoost. Subclass-level analysis is required to determine whether asymmetric features nevertheless benefited particular food categories.

In [69]:
# calculate metrics for every model-subclass combination
subclass_metric_records = []

grouping_columns = [
    "ClassDescription",
    "SubclassDescription",
    "Model",
    "Representation",
]

for group_values, group_data in forecast_evaluation_data.groupby(
    grouping_columns,
    sort=False,
):
    (
        class_description,
        subclass_description,
        model_name,
        representation,
    ) = group_values

    metrics = calculate_forecast_metrics(group_data)

    subclass_metric_records.append(
        {
            "ClassDescription": class_description,
            "SubclassDescription": subclass_description,
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

subclass_model_results = pd.DataFrame(subclass_metric_records)

print(
    "Subclass-model results:",
    len(subclass_model_results),
)
print(
    "Expected results:",
    46 * 7,
)
print(
    "Observations per result:",
    sorted(
        subclass_model_results["Observations"].unique()
    ),
)

Subclass-model results: 322
Expected results: 322
Observations per result: [np.int64(12)]


In [70]:
# identify the lowest-RMSE model for each food subclass
subclass_winners = (
    subclass_model_results.sort_values(
        [
            "SubclassDescription",
            "RMSE",
            "MAE",
        ]
    )
    .drop_duplicates(
        subset=["SubclassDescription"],
        keep="first",
    )
    .rename(
        columns={
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    [
        [
            "ClassDescription",
            "SubclassDescription",
            "Winning_Model",
            "Winning_Representation",
            "Winning_MAE",
            "Winning_RMSE",
            "Winning_R2",
            "Winning_Directional_Accuracy_Pct",
        ]
    ]
    .reset_index(drop=True)
)

subclass_winner_summary = (
    subclass_winners.groupby(
        [
            "Winning_Model",
            "Winning_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        "Food_Subclasses",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(subclass_winner_summary)
display(
    subclass_winners.sort_values(
        "Winning_RMSE"
    ).head(10)
)

,Winning_Model,Winning_Representation,Food_Subclasses
0,Random Forest,Symmetric,16
1,XGBoost,Symmetric,7
2,Persistence,Lag-1 benchmark,5
3,Random Forest,Asymmetric,5
4,XGBoost,Asymmetric,5
5,Ridge,Symmetric,4
6,Ridge,Asymmetric,4


,ClassDescription,SubclassDescription,Winning_Model,Winning_Representation,Winning_MAE,Winning_RMSE,Winning_R2,Winning_Directional_Accuracy_Pct
35,Soft drinks,Soft drinks,Persistence,Lag-1 benchmark,0.241626,0.294418,-0.250721,50.000000
0,Other food,Baby food,Persistence,Lag-1 benchmark,0.286480,0.339117,-1.241370,41.666667
12,Fruits and nuts,Fruit and nuts ground and other preparations,XGBoost,Symmetric,0.309381,0.372183,0.355309,75.000000
9,Fish and other seafood,Fish,Random Forest,Symmetric,0.314600,0.383227,-0.051256,75.000000
26,Other food,Other food products n.e.c.,Random Forest,Asymmetric,0.281066,0.408675,-0.220225,75.000000
1,Cereal products,Bread and bakery products,Ridge,Symmetric,0.315104,0.409537,-0.424188,66.666667
42,Vegetables,Vegetables and pulses ground and other prepara...,XGBoost,Symmetric,0.368402,0.434867,-0.154845,58.333333
17,Cereal products,"Macaroni, noodles, couscous and similar pasta ...",XGBoost,Asymmetric,0.348134,0.460431,-0.516696,75.000000
5,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",XGBoost,Asymmetric,0.394256,0.498729,0.015257,83.333333
44,Water,Water,XGBoost,Symmetric,0.432528,0.524496,-0.036607,66.666667


In [71]:
# compare symmetric and asymmetric RMSE by subclass
ml_subclass_results = subclass_model_results.loc[
    subclass_model_results["Model"] != "Persistence"
].copy()

subclass_representation_comparison = (
    ml_subclass_results.pivot(
        index=[
            "ClassDescription",
            "SubclassDescription",
            "Model",
        ],
        columns="Representation",
        values="RMSE",
    )
    .reset_index()
)

subclass_representation_comparison.columns.name = None

subclass_representation_comparison[
    "Asymmetric_Improvement_Pct"
] = (
    (
        subclass_representation_comparison["Symmetric"]
        - subclass_representation_comparison["Asymmetric"]
    )
    / subclass_representation_comparison["Symmetric"]
    * 100
)

subclass_representation_comparison[
    "Preferred_Representation"
] = np.where(
    subclass_representation_comparison["Asymmetric"]
    < subclass_representation_comparison["Symmetric"],
    "Asymmetric",
    "Symmetric",
)

subclass_representation_summary = (
    subclass_representation_comparison.groupby(
        [
            "Model",
            "Preferred_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        ["Model", "Preferred_Representation"]
    )
    .reset_index(drop=True)
)

display(subclass_representation_summary)

print("Largest asymmetric improvements:")
display(
    subclass_representation_comparison.sort_values(
        "Asymmetric_Improvement_Pct",
        ascending=False,
    ).head(10)
)

,Model,Preferred_Representation,Food_Subclasses
0,Random Forest,Asymmetric,23
1,Random Forest,Symmetric,23
2,Ridge,Asymmetric,11
3,Ridge,Symmetric,35
4,XGBoost,Asymmetric,16
5,XGBoost,Symmetric,30


Largest asymmetric improvements:


,ClassDescription,SubclassDescription,Model,Asymmetric,Symmetric,Asymmetric_Improvement_Pct,Preferred_Representation
96,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",Random Forest,0.505572,0.575587,12.164121,Asymmetric
129,Vegetables,Vegetables and pulses ground and other prepara...,Random Forest,0.464160,0.525447,11.663939,Asymmetric
12,Cereal products,"Macaroni, noodles, couscous and similar pasta ...",Random Forest,0.498739,0.562427,11.323772,Asymmetric
86,Other food,"Salt, condiments and sauces",XGBoost,0.662000,0.742925,10.892848,Asymmetric
93,Soft drinks,Soft drinks,Random Forest,0.400876,0.445238,9.963738,Asymmetric
65,"Milk, other dairy products and eggs",Other milk and cream,XGBoost,0.739107,0.812784,9.064772,Asymmetric
87,Other food,"Spices, culinary herbs and seeds",Random Forest,0.927382,1.019095,8.999465,Asymmetric
71,Oils and fats,Margarine and similar preparations,XGBoost,0.789302,0.844154,6.497936,Asymmetric
113,Tea,Tea and other plant products for infusion,XGBoost,0.769550,0.816799,5.784706,Asymmetric
63,"Milk, other dairy products and eggs",Other milk and cream,Random Forest,0.785311,0.829602,5.338785,Asymmetric


### Interpretation of subclass-level performance

Machine-learning models achieved the lowest subclass RMSE in 41 of the 46 food subclasses. Persistence was the strongest forecast in only five subclasses, showing that machine learning generally added predictive value beyond the previous month's inflation rate.

Symmetric Random Forest was the most frequent subclass winner, ranking first in 16 categories. Symmetric XGBoost won seven categories, while the remaining categories were distributed across asymmetric tree models, Ridge models, and persistence. Thus, the best pooled model was not automatically the best model for every food category.

Symmetric specifications were the winning forecasts for 27 subclasses,
asymmetric specifications for 14, and persistence for five. Within each
algorithm, symmetric features were preferred for 35 of 46 Ridge comparisons and 30 of 46 XGBoost comparisons. Random Forest was evenly divided, with each representation preferred for 23 subclasses.

Across all three algorithms, asymmetric features reduced subclass RMSE in 50 of 138 comparisons. Their forecasting value was therefore category-specific rather than universal. The largest asymmetric improvements were approximately 12.2% for chocolate and cocoa products under Random Forest, 11.7% for prepared vegetable and pulse products, and 11.3% for pasta products.

These subclass winners are retrospective test-period findings and are not used to refit or select new models. Furthermore, each subclass result is based on only 12 observations, so individual rankings and R² values should be interpreted cautiously.

In [72]:
# calculate metrics for each month and model
monthly_metric_records = []

for (
    forecast_month,
    model_name,
    representation,
), monthly_data in forecast_evaluation_data.groupby(
    ["Date", "Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(monthly_data)

    monthly_metric_records.append(
        {
            "Date": forecast_month,
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

monthly_model_results = pd.DataFrame(monthly_metric_records)

print("Monthly model results:", len(monthly_model_results))
print("Expected results:", 12 * 7)
print(
    "Observations per monthly result:",
    sorted(monthly_model_results["Observations"].unique()),
)

Monthly model results: 84
Expected results: 84
Observations per monthly result: [np.int64(46)]


In [73]:
# identify the lowest-RMSE forecast in each month
monthly_winners = (
    monthly_model_results.sort_values(
        ["Date", "RMSE", "MAE"]
    )
    .drop_duplicates(subset=["Date"], keep="first")
    .rename(
        columns={
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    [
        [
            "Date",
            "Winning_Model",
            "Winning_Representation",
            "Winning_MAE",
            "Winning_RMSE",
            "Winning_R2",
            "Winning_Directional_Accuracy_Pct",
        ]
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

monthly_winner_summary = (
    monthly_winners.groupby(
        ["Winning_Model", "Winning_Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Months_Won"})
    .sort_values("Months_Won", ascending=False)
    .reset_index(drop=True)
)

display(monthly_winners)
display(monthly_winner_summary)

,Date,Winning_Model,Winning_Representation,Winning_MAE,Winning_RMSE,Winning_R2,Winning_Directional_Accuracy_Pct
0,2025-01-01,Random Forest,Symmetric,1.199331,1.932682,0.035610,54.347826
1,2025-02-01,XGBoost,Symmetric,0.864124,1.194605,0.411526,71.739130
2,2025-03-01,Random Forest,Asymmetric,1.015759,1.752533,0.032738,67.391304
3,2025-04-01,Random Forest,Symmetric,1.079214,2.032790,0.115870,65.217391
4,2025-05-01,Persistence,Lag-1 benchmark,1.146359,1.640086,0.542885,47.826087
5,2025-06-01,Random Forest,Asymmetric,0.927133,1.481960,0.178227,67.391304
6,2025-07-01,Persistence,Lag-1 benchmark,1.244300,1.713075,0.502467,56.521739
7,2025-08-01,XGBoost,Symmetric,1.006774,1.554883,0.105164,45.652174
8,2025-09-01,Persistence,Lag-1 benchmark,0.961552,1.566576,0.526864,71.739130
9,2025-10-01,Persistence,Lag-1 benchmark,1.108263,1.764813,0.398771,54.347826


,Winning_Model,Winning_Representation,Months_Won
0,Persistence,Lag-1 benchmark,4
1,Random Forest,Symmetric,4
2,Random Forest,Asymmetric,2
3,XGBoost,Symmetric,2


In [74]:
# describe monthly inflation dispersion and forecast difficulty
monthly_actual_summary = (
    test_actuals.groupby("Date", as_index=False)
    .agg(
        Mean_Actual_Inflation=(
            "Food_Inflation_Pct",
            "mean",
        ),
        Actual_Inflation_Std=(
            "Food_Inflation_Pct",
            "std",
        ),
        Minimum_Actual_Inflation=(
            "Food_Inflation_Pct",
            "min",
        ),
        Maximum_Actual_Inflation=(
            "Food_Inflation_Pct",
            "max",
        ),
    )
)

monthly_difficulty = monthly_actual_summary.merge(
    monthly_winners[
        [
            "Date",
            "Winning_Model",
            "Winning_Representation",
            "Winning_RMSE",
        ]
    ],
    on="Date",
    how="left",
    validate="one_to_one",
)

monthly_difficulty["Date"] = (
    monthly_difficulty["Date"].dt.strftime("%Y-%m")
)

display(
    monthly_difficulty.sort_values(
        "Winning_RMSE",
        ascending=False,
    )
)

,Date,Mean_Actual_Inflation,Actual_Inflation_Std,Minimum_Actual_Inflation,Maximum_Actual_Inflation,Winning_Model,Winning_Representation,Winning_RMSE
10,2025-11,0.301289,2.346504,-8.111348,12.188982,Random Forest,Symmetric,2.160644
3,2025-04,0.596938,2.185783,-4.299728,11.442515,Random Forest,Symmetric,2.032790
0,2025-01,-0.084558,1.989788,-8.773891,5.164323,Random Forest,Symmetric,1.932682
9,2025-10,-0.042564,2.301185,-10.178411,5.070213,Persistence,Lag-1 benchmark,1.764813
2,2025-03,0.316300,1.801635,-5.654528,6.960291,Random Forest,Asymmetric,1.752533
6,2025-07,-0.161781,2.455489,-12.438372,6.057494,Persistence,Lag-1 benchmark,1.713075
4,2025-05,0.691416,2.452599,-3.005558,14.584680,Persistence,Lag-1 benchmark,1.640086
8,2025-09,-0.420810,2.302667,-12.977642,1.429981,Persistence,Lag-1 benchmark,1.566576
7,2025-08,0.102587,1.661876,-6.997019,2.317484,XGBoost,Symmetric,1.554883
5,2025-06,0.116058,1.652849,-6.891490,3.447723,Random Forest,Asymmetric,1.481960


### Interpretation of monthly performance

Machine-learning models achieved the lowest RMSE in eight of the twelve test months, while persistence was preferred in May, July, September, and October. This indicates that recent observed inflation sometimes provided a strong short-term benchmark.

Random Forest was the most frequent monthly winner. Its symmetric representation won four months and its asymmetric representation won two. Symmetric XGBoost won February and August.

Although symmetric XGBoost won only two individual months, it achieved the lowest RMSE across the complete test year. Overall RMSE depends on the magnitude of every forecast error rather than the number of monthly wins. A model can therefore rank first overall by avoiding especially large errors without being the lowest-error model in most individual months.

November, April, and January were the most difficult forecast months based on the lowest attainable monthly RMSE. February and December were comparatively easier. Monthly performance therefore varied substantially, supporting the use of a full twelve-month evaluation instead of relying on isolated periods.

In [75]:
# inspect available econometric data and forecast outputs
econometric_data_path = (
    processed_data_directory / "econometric_model_data.csv"
)

econometric_data = pd.read_csv(
    econometric_data_path,
    parse_dates=["Date"],
)

data_column_inventory = pd.concat(
    [
        pd.DataFrame(
            {
                "Dataset": "Econometric",
                "Column": econometric_data.columns,
            }
        ),
        pd.DataFrame(
            {
                "Dataset": "Machine learning",
                "Column": ml_data.columns,
            }
        ),
    ],
    ignore_index=True,
)

reports_table_directory = Path("../reports/tables")
existing_csv_files = sorted(
    reports_table_directory.rglob("*.csv")
)

var_or_forecast_files = [
    file_path
    for file_path in existing_csv_files
    if (
        "var" in file_path.stem.lower()
        or "forecast" in file_path.stem.lower()
    )
]

econometric_input_summary = pd.DataFrame(
    {
        "Value": [
            len(econometric_data),
            econometric_data[
                "SubclassDescription"
            ].nunique(),
            econometric_data["Date"].nunique(),
            econometric_data["Date"].min(),
            econometric_data["Date"].max(),
            int(econometric_data.isna().sum().sum()),
        ]
    },
    index=[
        "Observations",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Missing values",
    ],
)

display(econometric_input_summary)
display(data_column_inventory)

print("Existing VAR or forecast-related tables:")

if var_or_forecast_files:
    for file_path in var_or_forecast_files:
        print(file_path.as_posix())
else:
    print("None found.")

,Value
Observations,4830
Food subclasses,46
Unique months,105
Start date,2017-04-01 00:00:00
End date,2025-12-01 00:00:00
Missing values,0


,Dataset,Column
0,Econometric,Date
1,Econometric,GroupDescription
2,Econometric,ClassDescription
3,Econometric,SubclassDescription
4,Econometric,Subclass_Weight
5,Econometric,CPI
6,Econometric,Log_CPI
7,Econometric,Food_Inflation_Pct
8,Econometric,ExchangeRate
9,Econometric,Log_ExchangeRate


Existing VAR or forecast-related tables:
None found.


In [76]:
# inspect archived VAR forecast results
var_archive_directory = Path("../reports/archive/var")

var_actual_vs_forecast = pd.read_csv(
    var_archive_directory / "var_actual_vs_forecast.csv"
)

var_forecast_comparison = pd.read_csv(
    var_archive_directory / "var_forecast_comparison.csv"
)

var_file_summary = pd.DataFrame(
    [
        {
            "File": "var_actual_vs_forecast.csv",
            "Rows": len(var_actual_vs_forecast),
            "Columns": len(var_actual_vs_forecast.columns),
            "Missing_Values": int(
                var_actual_vs_forecast.isna().sum().sum()
            ),
        },
        {
            "File": "var_forecast_comparison.csv",
            "Rows": len(var_forecast_comparison),
            "Columns": len(var_forecast_comparison.columns),
            "Missing_Values": int(
                var_forecast_comparison.isna().sum().sum()
            ),
        },
    ]
)

var_column_inventory = pd.DataFrame(
    [
        {
            "File": "var_actual_vs_forecast.csv",
            "Column": column,
        }
        for column in var_actual_vs_forecast.columns
    ]
    + [
        {
            "File": "var_forecast_comparison.csv",
            "Column": column,
        }
        for column in var_forecast_comparison.columns
    ]
)

display(var_file_summary)
display(var_column_inventory)

print("VAR actual-versus-forecast preview:")
display(var_actual_vs_forecast.head(10))

print("VAR forecast-comparison table:")
display(var_forecast_comparison)

,File,Rows,Columns,Missing_Values
0,var_actual_vs_forecast.csv,12,2,0
1,var_forecast_comparison.csv,2830,2,0


,File,Column
0,var_actual_vs_forecast.csv,Actual CPI
1,var_actual_vs_forecast.csv,Forecast CPI
2,var_forecast_comparison.csv,Actual CPI
3,var_forecast_comparison.csv,Forecast CPI


VAR actual-versus-forecast preview:


,Actual CPI,Forecast CPI
0,101.000000,95.721322
1,100.600000,95.318089
2,100.900000,95.281598
3,100.600000,93.474067
4,101.800000,92.181029
5,101.700000,91.855426
6,102.700000,91.749538
7,103.400000,89.966545
8,103.400000,89.166436
9,102.700000,88.774040


VAR forecast-comparison table:


,Actual CPI,Forecast CPI
0,85.200000,83.709020
1,83.800000,82.487946
2,84.500000,82.742504
3,84.200000,84.119248
4,85.400000,83.739605
...,...,...
2825,103.400000,80.523885
2826,103.400000,80.523885
2827,102.700000,80.523885
2828,103.000000,80.523885


### Econometric forecast-benchmark design

No previously generated VAR or econometric forecast tables were found.
Consequently, forecast results cannot be inferred from the full-sample
cointegration models.

Separate short-run ARDL and NARDL forecast benchmarks are constructed for the comparison:

- the symmetric ARDL benchmark uses lagged food inflation and lagged symmetric exchange-rate changes
- the asymmetric NARDL benchmark uses lagged food inflation together with separate lagged depreciation and appreciation shocks
- monthly seasonal indicators are included
- candidate lag orders range from one to six
- all candidates use a common six-month hold-back
- BIC selection uses development data ending in December 2024
- predictions cover January–December 2025

Contemporaneous exchange-rate changes are excluded. This ensures that the econometric forecasts use lagged exchange-rate information consistent with the machine-learning forecasting exercise.

In [77]:
# construct lagged variables for econometric forecasting
maximum_forecast_lag = 6

econometric_forecast_data = (
    econometric_data.sort_values(
        ["SubclassDescription", "Date"]
    )
    .reset_index(drop=True)
    .copy()
)

lag_source_columns = [
    "Food_Inflation_Pct",
    "ExchangeRate_Log_Change_Pct",
    "Depreciation_Shock_Pct",
    "Appreciation_Shock_Pct",
]

for lag in range(1, maximum_forecast_lag + 1):
    for source_column in lag_source_columns:
        lagged_column = f"{source_column}_Lag{lag}"

        econometric_forecast_data[lagged_column] = (
            econometric_forecast_data.groupby(
                "SubclassDescription"
            )[source_column]
            .shift(lag)
        )


# Add monthly seasonal indicators
month_indicators = pd.get_dummies(
    econometric_forecast_data["Date"].dt.month,
    prefix="Month",
    drop_first=True,
    dtype=float,
)

econometric_forecast_data = pd.concat(
    [
        econometric_forecast_data,
        month_indicators,
    ],
    axis=1,
)

seasonal_columns = list(month_indicators.columns)

maximum_lag_columns = [
    f"{source_column}_Lag{maximum_forecast_lag}"
    for source_column in lag_source_columns
]

aligned_econometric_forecast_data = (
    econometric_forecast_data.dropna(
        subset=maximum_lag_columns
    )
    .copy()
)

forecast_development_data = (
    aligned_econometric_forecast_data.loc[
        aligned_econometric_forecast_data["Date"]
        <= pd.Timestamp("2024-12-01")
    ]
    .copy()
)

forecast_test_data = (
    aligned_econometric_forecast_data.loc[
        aligned_econometric_forecast_data["Date"].between(
            pd.Timestamp("2025-01-01"),
            pd.Timestamp("2025-12-01"),
        )
    ]
    .copy()
)

In [78]:
# validate the econometric forecasting sample
decomposition_error = np.abs(
    econometric_forecast_data[
        "ExchangeRate_Log_Change_Pct"
    ]
    - (
        econometric_forecast_data[
            "Depreciation_Shock_Pct"
        ]
        + econometric_forecast_data[
            "Appreciation_Shock_Pct"
        ]
    )
).max()

econometric_forecast_checks = pd.DataFrame(
    {
        "Check": [
            "Development sample contains 4,002 rows",
            "Development sample covers 87 months",
            "Development ends in December 2024",
            "Test sample contains 552 rows",
            "Test sample covers 12 months",
            "Test sample covers 46 subclasses",
            "No duplicate development rows",
            "No duplicate test rows",
            "No missing values in aligned data",
            "Depreciation shocks are non-negative",
            "Appreciation shocks are non-positive",
            "Shock decomposition is exact",
        ],
        "Passed": [
            len(forecast_development_data) == 4002,
            forecast_development_data["Date"].nunique() == 87,
            forecast_development_data["Date"].max()
            == pd.Timestamp("2024-12-01"),
            len(forecast_test_data) == 552,
            forecast_test_data["Date"].nunique() == 12,
            forecast_test_data[
                "SubclassDescription"
            ].nunique()
            == 46,
            not forecast_development_data.duplicated(
                identifier_columns
            ).any(),
            not forecast_test_data.duplicated(
                identifier_columns
            ).any(),
            not aligned_econometric_forecast_data.isna()
            .any()
            .any(),
            econometric_forecast_data[
                "Depreciation_Shock_Pct"
            ].min()
            >= 0,
            econometric_forecast_data[
                "Appreciation_Shock_Pct"
            ].max()
            <= 0,
            np.isclose(decomposition_error, 0.0),
        ],
    }
)

forecast_sample_summary = pd.DataFrame(
    [
        {
            "Sample": "Development",
            "Start_Date": forecast_development_data[
                "Date"
            ].min(),
            "End_Date": forecast_development_data[
                "Date"
            ].max(),
            "Observations": len(forecast_development_data),
            "Food_Subclasses": forecast_development_data[
                "SubclassDescription"
            ].nunique(),
            "Unique_Months": forecast_development_data[
                "Date"
            ].nunique(),
        },
        {
            "Sample": "Locked test",
            "Start_Date": forecast_test_data["Date"].min(),
            "End_Date": forecast_test_data["Date"].max(),
            "Observations": len(forecast_test_data),
            "Food_Subclasses": forecast_test_data[
                "SubclassDescription"
            ].nunique(),
            "Unique_Months": forecast_test_data[
                "Date"
            ].nunique(),
        },
    ]
)

display(econometric_forecast_checks)
display(forecast_sample_summary)

print(
    "Maximum shock-decomposition error:",
    decomposition_error,
)
print(
    "All econometric forecast checks passed:",
    bool(econometric_forecast_checks["Passed"].all()),
)

,Check,Passed
0,"Development sample contains 4,002 rows",True
1,Development sample covers 87 months,True
2,Development ends in December 2024,True
3,Test sample contains 552 rows,True
4,Test sample covers 12 months,True
5,Test sample covers 46 subclasses,True
6,No duplicate development rows,True
7,No duplicate test rows,True
8,No missing values in aligned data,True
9,Depreciation shocks are non-negative,True


,Sample,Start_Date,End_Date,Observations,Food_Subclasses,Unique_Months
0,Development,2017-10-01,2024-12-01,4002,46,87
1,Locked test,2025-01-01,2025-12-01,552,46,12


Maximum shock-decomposition error: 0.0
All econometric forecast checks passed: True


### Development only ARDL and NARDL lag selection

Separate models are estimated for each food subclass. Candidate food-inflation and exchange-rate lag orders range from one to six, producing 36 candidate specifications per subclass and model type.

All candidates use the same 87-month development sample after the common six-month hold-back. This ensures that BIC comparisons are not affected by different sample sizes.

The symmetric ARDL models include lagged exchange-rate changes. The asymmetric NARDL models replace these changes with separate lagged depreciation and appreciation shocks. Eleven monthly indicators control for seasonality.

No 2025 outcome is used for lag selection or coefficient estimation.

In [79]:
# estimate development-only lag candidates

def evaluate_lag_candidates(
    development_data,
    model_name,
    maximum_price_lag=6,
    maximum_exchange_lag=6,
):
    candidate_records = []
    failure_records = []

    for subclass, subclass_data in development_data.groupby(
        "SubclassDescription",
        sort=True,
    ):
        subclass_data = subclass_data.sort_values("Date")

        class_description = subclass_data[
            "ClassDescription"
        ].iloc[0]

        for price_lag in range(1, maximum_price_lag + 1):
            price_columns = [
                f"Food_Inflation_Pct_Lag{lag}"
                for lag in range(1, price_lag + 1)
            ]

            for exchange_lag in range(
                1,
                maximum_exchange_lag + 1,
            ):
                if model_name == "ARDL":
                    exchange_columns = [
                        (
                            "ExchangeRate_Log_Change_Pct"
                            f"_Lag{lag}"
                        )
                        for lag in range(
                            1,
                            exchange_lag + 1,
                        )
                    ]
                elif model_name == "NARDL":
                    exchange_columns = [
                        f"Depreciation_Shock_Pct_Lag{lag}"
                        for lag in range(
                            1,
                            exchange_lag + 1,
                        )
                    ] + [
                        f"Appreciation_Shock_Pct_Lag{lag}"
                        for lag in range(
                            1,
                            exchange_lag + 1,
                        )
                    ]
                else:
                    raise ValueError(
                        f"Unsupported model: {model_name}"
                    )

                predictor_columns = (
                    price_columns
                    + exchange_columns
                    + seasonal_columns
                )

                model_data = subclass_data[
                    ["Food_Inflation_Pct"]
                    + predictor_columns
                ].astype(float)

                design_matrix = sm.add_constant(
                    model_data[predictor_columns],
                    has_constant="add",
                )

                try:
                    fitted_model = sm.OLS(
                        model_data["Food_Inflation_Pct"],
                        design_matrix,
                    ).fit()

                    if not np.isfinite(fitted_model.bic):
                        raise ValueError("Non-finite BIC")

                    candidate_records.append(
                        {
                            "ClassDescription": (
                                class_description
                            ),
                            "SubclassDescription": subclass,
                            "Model": model_name,
                            "Price_Lag": price_lag,
                            "Exchange_Lag": exchange_lag,
                            "BIC": fitted_model.bic,
                            "AIC": fitted_model.aic,
                            "Observations": int(
                                fitted_model.nobs
                            ),
                            "Parameters": int(
                                len(fitted_model.params)
                            ),
                        }
                    )

                except Exception as error:
                    failure_records.append(
                        {
                            "SubclassDescription": subclass,
                            "Model": model_name,
                            "Price_Lag": price_lag,
                            "Exchange_Lag": exchange_lag,
                            "Error": str(error),
                        }
                    )

    return (
        pd.DataFrame(candidate_records),
        pd.DataFrame(failure_records),
    )

In [ ]:
# select the minimum-BIC model for each subclass
ardl_lag_candidates, ardl_lag_failures = (
    evaluate_lag_candidates(
        forecast_development_data,
        model_name="ARDL",
    )
)

nardl_lag_candidates, nardl_lag_failures = (
    evaluate_lag_candidates(
        forecast_development_data,
        model_name="NARDL",
    )
)

econometric_lag_candidates = pd.concat(
    [
        ardl_lag_candidates,
        nardl_lag_candidates,
    ],
    ignore_index=True,
)

econometric_lag_failures = pd.concat(
    [
        ardl_lag_failures,
        nardl_lag_failures,
    ],
    ignore_index=True,
)

successful_candidate_counts = (
    econometric_lag_candidates.groupby(
        ["SubclassDescription", "Model"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Successful_Candidates"})
)

selected_econometric_lags = (
    econometric_lag_candidates.sort_values(
        [
            "SubclassDescription",
            "Model",
            "BIC",
        ]
    )
    .drop_duplicates(
        subset=["SubclassDescription", "Model"],
        keep="first",
    )
    .merge(
        successful_candidate_counts,
        on=["SubclassDescription", "Model"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["Model", "SubclassDescription"]
    )
    .reset_index(drop=True)
)

candidate_validation = (
    successful_candidate_counts.groupby(
        "Model",
        as_index=False,
    )
    .agg(
        Food_Subclasses=(
            "SubclassDescription",
            "nunique",
        ),
        Minimum_Successful_Candidates=(
            "Successful_Candidates",
            "min",
        ),
        Maximum_Successful_Candidates=(
            "Successful_Candidates",
            "max",
        ),
    )
)

selected_lag_summary = (
    selected_econometric_lags.groupby(
        [
            "Model",
            "Price_Lag",
            "Exchange_Lag",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        [
            "Model",
            "Food_Subclasses",
            "Price_Lag",
            "Exchange_Lag",
        ],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

print(
    "Successful lag candidates:",
    len(econometric_lag_candidates),
)
print(
    "Expected lag candidates:",
    46 * 36 * 2,
)
print(
    "Failed lag candidates:",
    len(econometric_lag_failures),
)
print(
    "Selected econometric specifications:",
    len(selected_econometric_lags),
)

display(candidate_validation)
display(selected_lag_summary)

print(
    "All subclasses have 36 successful candidates:",
    bool(
        successful_candidate_counts[
            "Successful_Candidates"
        ].eq(36).all()
    ),
)

### Interpretation of forecast lag selection

Development-only BIC selection strongly favoured parsimonious forecast
specifications. An ARDL(1,1) structure was selected for 33 of 46 subclasses, while the equivalent NARDL structure was selected for 34 subclasses.

A one-month exchange-rate lag was selected for 40 ARDL models and 43 NARDL models. This suggests that additional exchange-rate lags rarely provided enough forecasting information to offset the BIC penalty for added parameters.

These forecasting specifications differ from the full-sample models in Notebook 06 because the present exercise excludes contemporaneous exchange-rate changes, uses only data available through December 2024, and targets out-of-sample food inflation rather than long-run inference.

In [ ]:
# refit selected models and forecast the locked test period
def get_econometric_predictors(
    model_name,
    price_lag,
    exchange_lag,
):
    price_columns = [
        f"Food_Inflation_Pct_Lag{lag}"
        for lag in range(1, price_lag + 1)
    ]

    if model_name == "ARDL":
        exchange_columns = [
            f"ExchangeRate_Log_Change_Pct_Lag{lag}"
            for lag in range(1, exchange_lag + 1)
        ]
    elif model_name == "NARDL":
        exchange_columns = [
            f"Depreciation_Shock_Pct_Lag{lag}"
            for lag in range(1, exchange_lag + 1)
        ] + [
            f"Appreciation_Shock_Pct_Lag{lag}"
            for lag in range(1, exchange_lag + 1)
        ]
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return price_columns + exchange_columns + seasonal_columns


econometric_prediction_frames = []
econometric_refit_records = []
fitted_econometric_forecast_models = {}

for _, selected_row in selected_econometric_lags.iterrows():
    subclass = selected_row["SubclassDescription"]
    model_name = selected_row["Model"]
    price_lag = int(selected_row["Price_Lag"])
    exchange_lag = int(selected_row["Exchange_Lag"])

    predictor_columns = get_econometric_predictors(
        model_name,
        price_lag,
        exchange_lag,
    )

    development_subset = (
        forecast_development_data.loc[
            forecast_development_data[
                "SubclassDescription"
            ]
            == subclass
        ]
        .sort_values("Date")
        .copy()
    )

    test_subset = (
        forecast_test_data.loc[
            forecast_test_data["SubclassDescription"]
            == subclass
        ]
        .sort_values("Date")
        .copy()
    )

    development_design = sm.add_constant(
        development_subset[predictor_columns].astype(float),
        has_constant="add",
    )

    test_design = sm.add_constant(
        test_subset[predictor_columns].astype(float),
        has_constant="add",
    )

    fitted_model = sm.OLS(
        development_subset["Food_Inflation_Pct"].astype(float),
        development_design,
    ).fit()

    predicted_values = fitted_model.predict(test_design)

    fitted_econometric_forecast_models[
        (model_name, subclass)
    ] = fitted_model

    representation = (
        "Symmetric"
        if model_name == "ARDL"
        else "Asymmetric"
    )

    prediction_frame = test_subset[
        identifier_columns
    ].copy()

    prediction_frame["Model"] = model_name
    prediction_frame["Representation"] = representation
    prediction_frame[
        "Predicted_Food_Inflation_Pct"
    ] = predicted_values.to_numpy()
    prediction_frame["Price_Lag"] = price_lag
    prediction_frame["Exchange_Lag"] = exchange_lag

    econometric_prediction_frames.append(prediction_frame)

    econometric_refit_records.append(
        {
            "SubclassDescription": subclass,
            "Model": model_name,
            "Price_Lag": price_lag,
            "Exchange_Lag": exchange_lag,
            "Selected_BIC": selected_row["BIC"],
            "Refitted_BIC": fitted_model.bic,
            "Absolute_BIC_Difference": abs(
                selected_row["BIC"] - fitted_model.bic
            ),
            "Development_Observations": int(
                fitted_model.nobs
            ),
            "Test_Predictions": len(predicted_values),
        }
    )

econometric_test_predictions = pd.concat(
    econometric_prediction_frames,
    ignore_index=True,
)

econometric_refit_validation = pd.DataFrame(
    econometric_refit_records
)

In [ ]:
# validate and evaluate econometric forecasts
econometric_prediction_summary = (
    econometric_test_predictions.groupby(
        ["Model", "Representation"],
        as_index=False,
    )
    .agg(
        Predictions=(
            "Predicted_Food_Inflation_Pct",
            "size",
        ),
        Food_Subclasses=(
            "SubclassDescription",
            "nunique",
        ),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
    )
)

econometric_forecast_validation = pd.DataFrame(
    {
        "Check": [
            "Ninety-two models were refitted",
            "Every refitted model used 87 observations",
            "Every refitted model produced 12 predictions",
            "Selected BIC values were reproduced",
            "Econometric prediction table has 1,104 rows",
            "Every model has 552 predictions",
            "Predictions contain no missing values",
            "Predictions are finite",
            "Prediction records are unique",
        ],
        "Passed": [
            len(econometric_refit_validation) == 92,
            econometric_refit_validation[
                "Development_Observations"
            ].eq(87).all(),
            econometric_refit_validation[
                "Test_Predictions"
            ].eq(12).all(),
            econometric_refit_validation[
                "Absolute_BIC_Difference"
            ].le(1e-10).all(),
            len(econometric_test_predictions) == 1104,
            econometric_prediction_summary[
                "Predictions"
            ].eq(552).all(),
            not econometric_test_predictions[
                "Predicted_Food_Inflation_Pct"
            ].isna().any(),
            np.isfinite(
                econometric_test_predictions[
                    "Predicted_Food_Inflation_Pct"
                ]
            ).all(),
            not econometric_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
        ],
    }
)

econometric_evaluation_data = (
    econometric_test_predictions.merge(
        test_actuals,
        on=identifier_columns,
        how="left",
        validate="many_to_one",
    )
)

econometric_metric_records = []

for (
    model_name,
    representation,
), model_data in econometric_evaluation_data.groupby(
    ["Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(model_data)

    econometric_metric_records.append(
        {
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

econometric_test_model_results = pd.DataFrame(
    econometric_metric_records
).sort_values("RMSE")

display(econometric_forecast_validation)
display(econometric_prediction_summary)
display(econometric_test_model_results)

print(
    "All ARDL and NARDL forecast checks passed:",
    bool(econometric_forecast_validation["Passed"].all()),
)

,Check,Passed
0,Ninety-two models were refitted,True
1,Every refitted model used 87 observations,True
2,Every refitted model produced 12 predictions,True
3,Selected BIC values were reproduced,True
4,"Econometric prediction table has 1,104 rows",True
5,Every model has 552 predictions,True
6,Predictions contain no missing values,True
7,Predictions are finite,True
8,Prediction records are unique,True


,Model,Representation,Predictions,Food_Subclasses,Start_Date,End_Date
0,ARDL,Symmetric,552,46,2025-01-01,2025-12-01
1,NARDL,Asymmetric,552,46,2025-01-01,2025-12-01


,Model,Representation,Observations,MAE,RMSE,R2,Directional_Accuracy_Pct,Mean_Bias
0,ARDL,Symmetric,552,1.106859,1.841097,0.192124,59.963768,0.287660
1,NARDL,Asymmetric,552,1.106109,1.848766,0.185379,60.507246,0.319212


All ARDL and NARDL forecast checks passed: True


### ARDL and NARDL forecast performance

Both leakage-safe econometric forecasts outperformed the persistence benchmark and the leading machine learning model on pooled test RMSE.

The symmetric ARDL model achieved an RMSE of 1.841 and R² of 0.192. The
asymmetric NARDL model achieved an RMSE of 1.849 and R² of 0.185. NARDL had a marginally lower MAE, but ARDL performed better on the prespecified primary RMSE criterion.

The NARDL representation therefore did not produce a clear aggregate forecasting advantage, although its performance remained competitive. Both models had positive mean bias, indicating average overprediction during 2025.

These forecasts are distinct from the full sample inferential models. Their lag orders and coefficients were estimated exclusively from the development period, and all predictions used only lagged information.

In [ ]:
# select development only VAR lag orders
from statsmodels.tsa.api import VAR

var_columns = [
    "Food_Inflation_Pct",
    "ExchangeRate_Log_Change_Pct",
]

common_var_outcome_start = pd.Timestamp("2017-10-01")
var_development_end = pd.Timestamp("2024-12-01")
maximum_var_lag = 6

var_candidate_records = []

for subclass, subclass_data in econometric_data.groupby(
    "SubclassDescription",
    sort=True,
):
    subclass_data = subclass_data.sort_values("Date")
    class_description = subclass_data[
        "ClassDescription"
    ].iloc[0]

    for var_lag in range(1, maximum_var_lag + 1):
        candidate_start = (
            common_var_outcome_start
            - pd.DateOffset(months=var_lag)
        )

        candidate_data = subclass_data.loc[
            subclass_data["Date"].between(
                candidate_start,
                var_development_end,
            ),
            var_columns,
        ].astype(float)

        fitted_var = VAR(candidate_data).fit(
            maxlags=var_lag,
            ic=None,
            trend="c",
        )

        var_candidate_records.append(
            {
                "ClassDescription": class_description,
                "SubclassDescription": subclass,
                "VAR_Lag": var_lag,
                "BIC": fitted_var.bic,
                "AIC": fitted_var.aic,
                "Observations": int(fitted_var.nobs),
                "System_Parameters": int(
                    fitted_var.df_model
                    * fitted_var.neqs
                ),
            }
        )

var_lag_candidates = pd.DataFrame(var_candidate_records)

selected_var_lags = (
    var_lag_candidates.sort_values(
        ["SubclassDescription", "BIC"]
    )
    .drop_duplicates(
        subset=["SubclassDescription"],
        keep="first",
    )
    .sort_values("SubclassDescription")
    .reset_index(drop=True)
)

var_lag_summary = (
    selected_var_lags.groupby(
        "VAR_Lag",
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values("VAR_Lag")
)

var_candidate_validation = pd.DataFrame(
    {
        "Check": [
            "All 276 VAR candidates were estimated",
            "Every candidate used 87 outcomes",
            "Forty-six VAR specifications were selected",
            "Every subclass has six candidates",
            "All BIC values are finite",
        ],
        "Passed": [
            len(var_lag_candidates) == 46 * 6,
            var_lag_candidates[
                "Observations"
            ].eq(87).all(),
            len(selected_var_lags) == 46,
            var_lag_candidates.groupby(
                "SubclassDescription"
            ).size().eq(6).all(),
            np.isfinite(
                var_lag_candidates["BIC"]
            ).all(),
        ],
    }
)

display(var_candidate_validation)
display(var_lag_summary)
display(selected_var_lags.head(10))

print(
    "All VAR lag-selection checks passed:",
    bool(var_candidate_validation["Passed"].all()),
)

,Check,Passed
0,All 276 VAR candidates were estimated,True
1,Every candidate used 87 outcomes,True
2,Forty-six VAR specifications were selected,True
3,Every subclass has six candidates,True
4,All BIC values are finite,True


,VAR_Lag,Food_Subclasses
0,1,46


,ClassDescription,SubclassDescription,VAR_Lag,BIC,AIC,Observations,System_Parameters
0,Other food,Baby food,1,2.900115,2.730053,87,6
1,Cereal products,Bread and bakery products,1,2.073851,1.903788,87,6
2,Cereal products,Breakfast cereals,1,3.204917,3.034854,87,6
3,Cereal products,Cereals,1,3.621039,3.450977,87,6
4,"Milk, other dairy products and eggs",Cheese,1,2.731903,2.561840,87,6
5,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",1,2.520343,2.350280,87,6
6,Coffee,Coffee and coffee substitutes,1,3.652578,3.482515,87,6
7,Fruits and nuts,"Dates, figs and tropical fruits, fresh",1,5.849168,5.679106,87,6
8,"Milk, other dairy products and eggs",Eggs,1,3.993147,3.823085,87,6
9,Fish and other seafood,Fish,1,2.045793,1.875730,87,6


All VAR lag-selection checks passed: True


In [ ]:
# generate rolling one month ahead VAR forecasts
var_prediction_records = []
var_refit_records = []
fitted_var_models = {}

for _, selected_row in selected_var_lags.iterrows():
    subclass = selected_row["SubclassDescription"]
    selected_lag = int(selected_row["VAR_Lag"])

    subclass_data = (
        econometric_data.loc[
            econometric_data["SubclassDescription"]
            == subclass
        ]
        .sort_values("Date")
        .copy()
    )

    estimation_start = (
        common_var_outcome_start
        - pd.DateOffset(months=selected_lag)
    )

    estimation_data = subclass_data.loc[
        subclass_data["Date"].between(
            estimation_start,
            var_development_end,
        ),
        var_columns,
    ].astype(float)

    fitted_var = VAR(estimation_data).fit(
        maxlags=selected_lag,
        ic=None,
        trend="c",
    )

    fitted_var_models[subclass] = fitted_var

    test_subset = forecast_test_data.loc[
        forecast_test_data["SubclassDescription"]
        == subclass
    ].sort_values("Date")

    for _, test_row in test_subset.iterrows():
        forecast_date = test_row["Date"]

        available_history = subclass_data.loc[
            subclass_data["Date"] < forecast_date,
            var_columns,
        ].tail(selected_lag)

        if len(available_history) != selected_lag:
            raise ValueError(
                f"Insufficient VAR history for {subclass} "
                f"at {forecast_date:%Y-%m}."
            )

        forecast_values = fitted_var.forecast(
            available_history.to_numpy(dtype=float),
            steps=1,
        )[0]

        var_prediction_records.append(
            {
                "Date": forecast_date,
                "ClassDescription": test_row[
                    "ClassDescription"
                ],
                "SubclassDescription": subclass,
                "Model": "VAR",
                "Representation": "Bivariate system",
                "Predicted_Food_Inflation_Pct": float(
                    forecast_values[0]
                ),
                "VAR_Lag": selected_lag,
            }
        )

    var_refit_records.append(
        {
            "SubclassDescription": subclass,
            "VAR_Lag": selected_lag,
            "Selected_BIC": selected_row["BIC"],
            "Refitted_BIC": fitted_var.bic,
            "Absolute_BIC_Difference": abs(
                selected_row["BIC"] - fitted_var.bic
            ),
            "Development_Observations": int(
                fitted_var.nobs
            ),
            "Test_Predictions": len(test_subset),
        }
    )

var_test_predictions = pd.DataFrame(
    var_prediction_records
)

var_refit_validation = pd.DataFrame(
    var_refit_records
)

In [ ]:
# validate and evaluate the VAR benchmark
var_forecast_checks = pd.DataFrame(
    {
        "Check": [
            "Forty-six VAR models were refitted",
            "Every model used 87 observations",
            "Selected BIC values were reproduced",
            "Every model produced 12 predictions",
            "VAR prediction table contains 552 rows",
            "Predictions contain no missing values",
            "Predictions are finite",
            "Prediction records are unique",
        ],
        "Passed": [
            len(var_refit_validation) == 46,
            var_refit_validation[
                "Development_Observations"
            ].eq(87).all(),
            var_refit_validation[
                "Absolute_BIC_Difference"
            ].le(1e-10).all(),
            var_refit_validation[
                "Test_Predictions"
            ].eq(12).all(),
            len(var_test_predictions) == 552,
            not var_test_predictions[
                "Predicted_Food_Inflation_Pct"
            ].isna().any(),
            np.isfinite(
                var_test_predictions[
                    "Predicted_Food_Inflation_Pct"
                ]
            ).all(),
            not var_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
        ],
    }
)

var_evaluation_data = var_test_predictions.merge(
    test_actuals,
    on=identifier_columns,
    how="left",
    validate="many_to_one",
)

var_test_model_results = pd.DataFrame(
    [
        {
            "Model": "VAR",
            "Representation": "Bivariate system",
            **calculate_forecast_metrics(
                var_evaluation_data
            ),
        }
    ]
)

display(var_forecast_checks)
display(var_test_model_results)

print(
    "All VAR forecast checks passed:",
    bool(var_forecast_checks["Passed"].all()),
)

,Check,Passed
0,Forty-six VAR models were refitted,True
1,Every model used 87 observations,True
2,Selected BIC values were reproduced,True
3,Every model produced 12 predictions,True
4,VAR prediction table contains 552 rows,True
5,Predictions contain no missing values,True
6,Predictions are finite,True
7,Prediction records are unique,True


,Model,Representation,Observations,MAE,RMSE,R2,Directional_Accuracy_Pct,Mean_Bias
0,VAR,Bivariate system,552,1.061515,1.895828,0.143378,62.862319,0.297821


All VAR forecast checks passed: True


### Interpretation of VAR performance

BIC selected a one lag VAR for every food subclass. This consistent result indicates that additional monthly dynamics did not provide sufficient forecasting value to justify their added parameters.

The leakage safe VAR achieved an RMSE of 1.896, MAE of 1.062, and R² of 0.143. Its RMSE was higher than those of ARDL and NARDL but lower than that of the leading machine-learning model.

VAR performed particularly well on MAE and directional accuracy. Its MAE was almost identical to symmetric XGBoost, while its directional accuracy of 62.86% was higher than those of ARDL, NARDL, and XGBoost.

The new results are directly comparable because they forecast the same food inflation target, cover the same 552 subclass-month outcomes, and use development data ending in December 2024.

In [ ]:
# combine all locked-test forecasts
common_prediction_columns = (
    identifier_columns
    + [
        "Model",
        "Representation",
        "Predicted_Food_Inflation_Pct",
    ]
)

all_test_predictions = pd.concat(
    [
        ml_test_predictions[common_prediction_columns],
        econometric_test_predictions[
            common_prediction_columns
        ],
        var_test_predictions[common_prediction_columns],
    ],
    ignore_index=True,
)

all_prediction_counts = (
    all_test_predictions.groupby(
        ["Model", "Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Predictions"})
)

combined_forecast_checks = pd.DataFrame(
    {
        "Check": [
            "Ten forecast variants are present",
            "Every variant has 552 predictions",
            "Combined table contains 5,520 rows",
            "Predictions contain no missing values",
            "Predictions are finite",
            "Prediction records are unique",
        ],
        "Passed": [
            len(all_prediction_counts) == 10,
            all_prediction_counts[
                "Predictions"
            ].eq(552).all(),
            len(all_test_predictions) == 10 * 552,
            not all_test_predictions[
                "Predicted_Food_Inflation_Pct"
            ].isna().any(),
            np.isfinite(
                all_test_predictions[
                    "Predicted_Food_Inflation_Pct"
                ]
            ).all(),
            not all_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
        ],
    }
)

all_forecast_evaluation_data = all_test_predictions.merge(
    test_actuals,
    on=identifier_columns,
    how="left",
    validate="many_to_one",
)

display(combined_forecast_checks)
display(all_prediction_counts)

print(
    "All combined forecast checks passed:",
    bool(combined_forecast_checks["Passed"].all()),
)

,Check,Passed
0,Ten forecast variants are present,True
1,Every variant has 552 predictions,True
2,"Combined table contains 5,520 rows",True
3,Predictions contain no missing values,True
4,Predictions are finite,True
5,Prediction records are unique,True


,Model,Representation,Predictions
0,ARDL,Symmetric,552
1,NARDL,Asymmetric,552
2,Persistence,Lag-1 benchmark,552
3,Random Forest,Asymmetric,552
4,Random Forest,Symmetric,552
5,Ridge,Asymmetric,552
6,Ridge,Symmetric,552
7,VAR,Bivariate system,552
8,XGBoost,Asymmetric,552
9,XGBoost,Symmetric,552


All combined forecast checks passed: True


In [ ]:
# calculate metrics and rankings for all models
all_model_metric_records = []

for (
    model_name,
    representation,
), model_data in all_forecast_evaluation_data.groupby(
    ["Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(model_data)

    if model_name == "Persistence":
        modelling_approach = "Benchmark"
    elif model_name in {
        "Ridge",
        "Random Forest",
        "XGBoost",
    }:
        modelling_approach = "Machine learning"
    else:
        modelling_approach = "Econometric"

    all_model_metric_records.append(
        {
            "Approach": modelling_approach,
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

all_test_model_results = pd.DataFrame(
    all_model_metric_records
)

combined_persistence_result = (
    all_test_model_results.loc[
        all_test_model_results["Model"] == "Persistence"
    ]
    .iloc[0]
)

all_test_model_results[
    "RMSE_Improvement_vs_Persistence_Pct"
] = (
    (
        combined_persistence_result["RMSE"]
        - all_test_model_results["RMSE"]
    )
    / combined_persistence_result["RMSE"]
    * 100
)

all_test_model_results[
    "MAE_Improvement_vs_Persistence_Pct"
] = (
    (
        combined_persistence_result["MAE"]
        - all_test_model_results["MAE"]
    )
    / combined_persistence_result["MAE"]
    * 100
)

all_test_model_results["RMSE_Rank"] = (
    all_test_model_results["RMSE"]
    .rank(method="min")
    .astype(int)
)

all_test_model_results["MAE_Rank"] = (
    all_test_model_results["MAE"]
    .rank(method="min")
    .astype(int)
)

all_test_model_results["R2_Rank"] = (
    all_test_model_results["R2"]
    .rank(method="min", ascending=False)
    .astype(int)
)

all_test_model_results["Directional_Rank"] = (
    all_test_model_results["Directional_Accuracy_Pct"]
    .rank(method="min", ascending=False)
    .astype(int)
)

all_test_model_results = (
    all_test_model_results.sort_values(
        ["RMSE_Rank", "MAE_Rank"]
    )
    .reset_index(drop=True)
)

display(
    all_test_model_results[
        [
            "RMSE_Rank",
            "MAE_Rank",
            "Directional_Rank",
            "Approach",
            "Model",
            "Representation",
            "MAE",
            "RMSE",
            "R2",
            "Directional_Accuracy_Pct",
            "Mean_Bias",
            "RMSE_Improvement_vs_Persistence_Pct",
            "MAE_Improvement_vs_Persistence_Pct",
        ]
    ]
)

,RMSE_Rank,MAE_Rank,Directional_Rank,Approach,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct,Mean_Bias,RMSE_Improvement_vs_Persistence_Pct,MAE_Improvement_vs_Persistence_Pct
0,1,9,8,Econometric,ARDL,Symmetric,1.106859,1.841097,0.192124,59.963768,0.287660,17.639557,16.972105
1,2,8,7,Econometric,NARDL,Asymmetric,1.106109,1.848766,0.185379,60.507246,0.319212,17.296478,17.028354
2,3,2,2,Econometric,VAR,Bivariate system,1.061515,1.895828,0.143378,62.862319,0.297821,15.191179,20.373496
3,4,1,9,Machine learning,XGBoost,Symmetric,1.060854,1.929075,0.113069,59.782609,0.263793,13.703894,20.423091
4,5,3,1,Machine learning,Random Forest,Symmetric,1.066689,1.964122,0.080550,63.405797,0.314879,12.136122,19.985359
5,6,6,3,Machine learning,XGBoost,Asymmetric,1.090656,1.973192,0.072038,61.231884,0.323877,11.730351,18.187517
6,7,5,6,Machine learning,Ridge,Symmetric,1.089515,1.984985,0.060913,60.869565,0.280788,11.202815,18.273135
7,8,7,5,Machine learning,Ridge,Asymmetric,1.092858,1.986590,0.059394,61.050725,0.287840,11.131018,18.022346
8,9,4,3,Machine learning,Random Forest,Asymmetric,1.075816,1.991030,0.055185,61.231884,0.313877,10.932393,19.300687
9,10,10,10,Benchmark,Persistence,Lag-1 benchmark,1.333117,2.235414,-0.190988,54.347826,0.025886,0.000000,0.000000


In [ ]:
# identify the leader under each evaluation criterion
metric_leader_records = []

metric_rules = {
    "Lowest MAE": ("MAE", "min"),
    "Lowest RMSE": ("RMSE", "min"),
    "Highest R2": ("R2", "max"),
    "Highest directional accuracy": (
        "Directional_Accuracy_Pct",
        "max",
    ),
    "Lowest absolute bias": ("Absolute_Bias", "min"),
}

leader_data = all_test_model_results.copy()
leader_data["Absolute_Bias"] = (
    leader_data["Mean_Bias"].abs()
)

for criterion, (metric_column, rule) in metric_rules.items():
    if rule == "min":
        leader_row = leader_data.loc[
            leader_data[metric_column].idxmin()
        ]
    else:
        leader_row = leader_data.loc[
            leader_data[metric_column].idxmax()
        ]

    metric_leader_records.append(
        {
            "Criterion": criterion,
            "Model": leader_row["Model"],
            "Representation": leader_row[
                "Representation"
            ],
            "Value": leader_row[metric_column],
        }
    )

metric_leaders = pd.DataFrame(metric_leader_records)

display(metric_leaders)

,Criterion,Model,Representation,Value
0,Lowest MAE,XGBoost,Symmetric,1.060854
1,Lowest RMSE,ARDL,Symmetric,1.841097
2,Highest R2,ARDL,Symmetric,0.192124
3,Highest directional accuracy,Random Forest,Symmetric,63.405797
4,Lowest absolute bias,Persistence,Lag-1 benchmark,0.025886


### Interpretation of the complete model comparison

The complete test comparison does not identify one model as superior under every criterion.

Symmetric ARDL ranked first on the primary RMSE criterion, with an RMSE of 1.841 and R² of 0.192. It reduced RMSE by 17.64% relative to persistence and outperformed symmetric XGBoost by approximately 4.56% on RMSE.

Symmetric XGBoost achieved the lowest MAE of 1.061, narrowly outperforming VAR. Its lower MAE indicates smaller typical forecast errors, whereas ARDL's lower RMSE indicates better protection against comparatively large errors.

Symmetric Random Forest achieved the highest directional accuracy at 63.41%. It was therefore most successful at identifying whether food inflation was positive or negative, even though it did not minimise forecast-error magnitude.

NARDL ranked second on RMSE but did not outperform symmetric ARDL. Similarly, the asymmetric machine learning specifications did not outperform their symmetric counterparts on pooled RMSE. The aggregate evidence therefore does not support a universal forecasting advantage from asymmetric exchange-rate decomposition.

Persistence had the lowest absolute bias but ranked last on MAE, RMSE, R², and directional accuracy. Low average bias alone therefore did not indicate strong forecast performance because positive and negative errors could cancel each other.

Overall, the results support complementary strengths rather than universal machine learning dominance: ARDL performed best on large-error-sensitive criteria, XGBoost on typical absolute error, and Random Forest on directional accuracy.

In [ ]:
# calculate subclass metrics across all ten variants
def classify_modelling_approach(model_name):
    if model_name == "Persistence":
        return "Benchmark"

    if model_name in {
        "Ridge",
        "Random Forest",
        "XGBoost",
    }:
        return "Machine learning"

    return "Econometric"


combined_subclass_records = []

for group_values, group_data in (
    all_forecast_evaluation_data.groupby(
        [
            "ClassDescription",
            "SubclassDescription",
            "Model",
            "Representation",
        ],
        sort=False,
    )
):
    (
        class_description,
        subclass_description,
        model_name,
        representation,
    ) = group_values

    metrics = calculate_forecast_metrics(group_data)

    combined_subclass_records.append(
        {
            "ClassDescription": class_description,
            "SubclassDescription": subclass_description,
            "Approach": classify_modelling_approach(
                model_name
            ),
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

combined_subclass_results = pd.DataFrame(
    combined_subclass_records
)

combined_subclass_winners = (
    combined_subclass_results.sort_values(
        [
            "SubclassDescription",
            "RMSE",
            "MAE",
        ]
    )
    .drop_duplicates(
        subset=["SubclassDescription"],
        keep="first",
    )
    .rename(
        columns={
            "Approach": "Winning_Approach",
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    .reset_index(drop=True)
)

combined_subclass_winner_summary = (
    combined_subclass_winners.groupby(
        [
            "Winning_Approach",
            "Winning_Model",
            "Winning_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        "Food_Subclasses",
        ascending=False,
    )
    .reset_index(drop=True)
)

subclass_approach_summary = (
    combined_subclass_winners.groupby(
        "Winning_Approach",
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        "Food_Subclasses",
        ascending=False,
    )
)

print(
    "Combined subclass results:",
    len(combined_subclass_results),
)
display(combined_subclass_winner_summary)
display(subclass_approach_summary)

Combined subclass results: 460


,Winning_Approach,Winning_Model,Winning_Representation,Food_Subclasses
0,Machine learning,Random Forest,Symmetric,8
1,Econometric,VAR,Bivariate system,8
2,Machine learning,XGBoost,Symmetric,7
3,Benchmark,Persistence,Lag-1 benchmark,5
4,Econometric,ARDL,Symmetric,4
5,Econometric,NARDL,Asymmetric,4
6,Machine learning,XGBoost,Asymmetric,3
7,Machine learning,Ridge,Symmetric,3
8,Machine learning,Random Forest,Asymmetric,2
9,Machine learning,Ridge,Asymmetric,2


,Winning_Approach,Food_Subclasses
2,Machine learning,25
1,Econometric,16
0,Benchmark,5


In [ ]:
# calculate monthly metrics across all ten variants
combined_monthly_records = []

for (
    forecast_month,
    model_name,
    representation,
), monthly_data in all_forecast_evaluation_data.groupby(
    ["Date", "Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(monthly_data)

    combined_monthly_records.append(
        {
            "Date": forecast_month,
            "Approach": classify_modelling_approach(
                model_name
            ),
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

combined_monthly_results = pd.DataFrame(
    combined_monthly_records
)

combined_monthly_winners = (
    combined_monthly_results.sort_values(
        ["Date", "RMSE", "MAE"]
    )
    .drop_duplicates(subset=["Date"], keep="first")
    .rename(
        columns={
            "Approach": "Winning_Approach",
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

combined_monthly_winner_summary = (
    combined_monthly_winners.groupby(
        [
            "Winning_Approach",
            "Winning_Model",
            "Winning_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Months_Won"})
    .sort_values(
        "Months_Won",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "Combined monthly results:",
    len(combined_monthly_results),
)
display(combined_monthly_winners)
display(combined_monthly_winner_summary)

Combined monthly results: 120


,Date,Winning_Approach,Winning_Model,Winning_Representation,Observations,Winning_MAE,Winning_RMSE,Winning_R2,Winning_Directional_Accuracy_Pct,Mean_Bias
0,2025-01-01,Machine learning,Random Forest,Symmetric,46,1.199331,1.932682,0.035610,54.347826,0.637169
1,2025-02-01,Econometric,ARDL,Symmetric,46,0.890094,1.143124,0.461154,71.739130,0.227097
2,2025-03-01,Econometric,ARDL,Symmetric,46,0.972162,1.385422,0.395528,60.869565,0.113934
3,2025-04-01,Econometric,ARDL,Symmetric,46,1.061006,1.692168,0.387342,54.347826,-0.270196
4,2025-05-01,Benchmark,Persistence,Lag-1 benchmark,46,1.146359,1.640086,0.542885,47.826087,-0.094478
5,2025-06-01,Econometric,NARDL,Asymmetric,46,1.067168,1.470251,0.191162,67.391304,0.188863
6,2025-07-01,Benchmark,Persistence,Lag-1 benchmark,46,1.244300,1.713075,0.502467,56.521739,0.277839
7,2025-08-01,Machine learning,XGBoost,Symmetric,46,1.006774,1.554883,0.105164,45.652174,-0.040290
8,2025-09-01,Benchmark,Persistence,Lag-1 benchmark,46,0.961552,1.566576,0.526864,71.739130,0.523397
9,2025-10-01,Benchmark,Persistence,Lag-1 benchmark,46,1.108263,1.764813,0.398771,54.347826,-0.378247


,Winning_Approach,Winning_Model,Winning_Representation,Months_Won
0,Benchmark,Persistence,Lag-1 benchmark,4
1,Econometric,ARDL,Symmetric,3
2,Econometric,NARDL,Asymmetric,2
3,Machine learning,Random Forest,Symmetric,2
4,Machine learning,XGBoost,Symmetric,1


### Interpretation of combined performance heterogeneity

Machine learning models produced the lowest RMSE for 25 of the 46 food
subclasses, compared with 16 econometric wins and five persistence wins. VAR and symmetric Random Forest were the most frequent individual subclass winners, each ranking first in eight categories.

This result does not conflict with ARDL's leading pooled RMSE. Pooled RMSE is especially sensitive to large errors. ARDL can therefore rank first overall by reducing severe errors in volatile categories even when another model performs better in a larger number of individual subclasses.

Performance also changed across time. Econometric models won five months, machine learning models won three, and persistence won four. ARDL performed best from February to April, while NARDL won June and November. Machine learning models led in January, August, and December.

No approach dominated all subclasses or months. The evidence instead supports model-dependent and category-dependent forecasting strengths.

In [ ]:
# define a HAC-adjusted loss-difference test
from statsmodels.stats.multitest import multipletests


def hac_loss_difference_test(
    loss_difference,
    hac_lag=1,
):
    loss_difference = np.asarray(
        loss_difference,
        dtype=float,
    )

    observations = len(loss_difference)
    mean_difference = loss_difference.mean()
    centred_difference = (
        loss_difference - mean_difference
    )

    long_run_variance = np.dot(
        centred_difference,
        centred_difference,
    ) / observations

    for lag in range(1, hac_lag + 1):
        autocovariance = np.dot(
            centred_difference[lag:],
            centred_difference[:-lag],
        ) / observations

        bartlett_weight = 1 - lag / (hac_lag + 1)

        long_run_variance += (
            2 * bartlett_weight * autocovariance
        )

    variance_of_mean = (
        long_run_variance / observations
    )

    if variance_of_mean <= 0:
        raise ValueError(
            "The estimated loss-difference variance "
            "is not positive."
        )

    test_statistic = (
        mean_difference
        / np.sqrt(variance_of_mean)
    )

    forecast_horizon = 1
    harvey_correction = np.sqrt(
        (
            observations
            + 1
            - 2 * forecast_horizon
            + (
                forecast_horizon
                * (forecast_horizon - 1)
                / observations
            )
        )
        / observations
    )

    corrected_statistic = (
        test_statistic * harvey_correction
    )

    p_value = 2 * stats.t.sf(
        np.abs(corrected_statistic),
        df=observations - 1,
    )

    return {
        "Observations": observations,
        "Mean_Loss_Difference": mean_difference,
        "HAC_Test_Statistic": corrected_statistic,
        "P_Value": p_value,
    }

In [ ]:
# compare monthly squared losses with ARDL
forecast_loss_data = (
    all_forecast_evaluation_data.copy()
)

forecast_loss_data["Forecast_Variant"] = (
    forecast_loss_data["Model"]
    + " | "
    + forecast_loss_data["Representation"]
)

forecast_loss_data["Squared_Error"] = (
    forecast_loss_data[
        "Predicted_Food_Inflation_Pct"
    ]
    - forecast_loss_data["Food_Inflation_Pct"]
) ** 2

monthly_loss = (
    forecast_loss_data.groupby(
        ["Date", "Forecast_Variant"],
        as_index=False,
    )["Squared_Error"]
    .mean()
    .pivot(
        index="Date",
        columns="Forecast_Variant",
        values="Squared_Error",
    )
    .sort_index()
)

reference_variant = "ARDL | Symmetric"
forecast_accuracy_records = []

for alternative_variant in monthly_loss.columns:
    if alternative_variant == reference_variant:
        continue

    loss_difference = (
        monthly_loss[alternative_variant]
        - monthly_loss[reference_variant]
    )

    test_result = hac_loss_difference_test(
        loss_difference,
        hac_lag=1,
    )

    alternative_model, alternative_representation = (
        alternative_variant.split(" | ", maxsplit=1)
    )

    alternative_rmse = all_test_model_results.loc[
        (
            all_test_model_results["Model"]
            == alternative_model
        )
        & (
            all_test_model_results["Representation"]
            == alternative_representation
        ),
        "RMSE",
    ].iloc[0]

    forecast_accuracy_records.append(
        {
            "Reference": reference_variant,
            "Alternative": alternative_variant,
            "Reference_RMSE": all_test_model_results.loc[
                all_test_model_results["Model"] == "ARDL",
                "RMSE",
            ].iloc[0],
            "Alternative_RMSE": alternative_rmse,
            **test_result,
        }
    )

forecast_accuracy_tests = pd.DataFrame(
    forecast_accuracy_records
)

forecast_accuracy_tests["Holm_Adjusted_P_Value"] = (
    multipletests(
        forecast_accuracy_tests["P_Value"],
        alpha=0.05,
        method="holm",
    )[1]
)

forecast_accuracy_tests["Lower_Loss_Model"] = np.where(
    forecast_accuracy_tests["Mean_Loss_Difference"] > 0,
    "ARDL",
    forecast_accuracy_tests["Alternative"],
)

forecast_accuracy_tests[
    "Significant_After_Holm"
] = (
    forecast_accuracy_tests[
        "Holm_Adjusted_P_Value"
    ]
    < 0.05
)

forecast_accuracy_tests = (
    forecast_accuracy_tests.sort_values(
        "P_Value"
    ).reset_index(drop=True)
)

display(forecast_accuracy_tests)

,Reference,Alternative,Reference_RMSE,Alternative_RMSE,Observations,Mean_Loss_Difference,HAC_Test_Statistic,P_Value,Holm_Adjusted_P_Value,Lower_Loss_Model,Significant_After_Holm
0,ARDL | Symmetric,Persistence | Lag-1 benchmark,1.841097,2.235414,12,1.607438,2.347776,0.038639,0.347748,ARDL,False
1,ARDL | Symmetric,Random Forest | Asymmetric,1.841097,1.991030,12,0.574562,2.325015,0.040221,0.347748,ARDL,False
2,ARDL | Symmetric,Ridge | Asymmetric,1.841097,1.986590,12,0.556901,2.272705,0.044098,0.347748,ARDL,False
3,ARDL | Symmetric,Ridge | Symmetric,1.841097,1.984985,12,0.550527,2.245647,0.046241,0.347748,ARDL,False
4,ARDL | Symmetric,Random Forest | Symmetric,1.841097,1.964122,12,0.468135,2.042268,0.065847,0.347748,ARDL,False
5,ARDL | Symmetric,XGBoost | Asymmetric,1.841097,1.973192,12,0.503849,1.708247,0.115619,0.462477,ARDL,False
6,ARDL | Symmetric,XGBoost | Symmetric,1.841097,1.929075,12,0.331694,1.080855,0.302889,0.908667,ARDL,False
7,ARDL | Symmetric,VAR | Bivariate system,1.841097,1.895828,12,0.204527,0.721909,0.485414,0.970827,ARDL,False
8,ARDL | Symmetric,NARDL | Asymmetric,1.841097,1.848766,12,0.028298,0.456847,0.656671,0.970827,ARDL,False


In [ ]:
# compare squared loss across the 46 subclasses
subclass_loss = (
    forecast_loss_data.groupby(
        [
            "SubclassDescription",
            "Forecast_Variant",
        ],
        as_index=False,
    )["Squared_Error"]
    .mean()
    .pivot(
        index="SubclassDescription",
        columns="Forecast_Variant",
        values="Squared_Error",
    )
)

subclass_accuracy_records = []

for alternative_variant in subclass_loss.columns:
    if alternative_variant == reference_variant:
        continue

    loss_difference = (
        subclass_loss[alternative_variant]
        - subclass_loss[reference_variant]
    )

    wilcoxon_result = stats.wilcoxon(
        loss_difference,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto",
    )

    subclass_accuracy_records.append(
        {
            "Reference": reference_variant,
            "Alternative": alternative_variant,
            "Food_Subclasses": len(loss_difference),
            "Median_Loss_Difference": (
                loss_difference.median()
            ),
            "Subclasses_ARDL_Better": int(
                (loss_difference > 0).sum()
            ),
            "Subclasses_Alternative_Better": int(
                (loss_difference < 0).sum()
            ),
            "Wilcoxon_Statistic": (
                wilcoxon_result.statistic
            ),
            "P_Value": wilcoxon_result.pvalue,
        }
    )

subclass_accuracy_tests = pd.DataFrame(
    subclass_accuracy_records
)

subclass_accuracy_tests[
    "Holm_Adjusted_P_Value"
] = multipletests(
    subclass_accuracy_tests["P_Value"],
    alpha=0.05,
    method="holm",
)[1]

subclass_accuracy_tests[
    "Significant_After_Holm"
] = (
    subclass_accuracy_tests[
        "Holm_Adjusted_P_Value"
    ]
    < 0.05
)

subclass_accuracy_tests = (
    subclass_accuracy_tests.sort_values(
        "P_Value"
    ).reset_index(drop=True)
)

display(subclass_accuracy_tests)

,Reference,Alternative,Food_Subclasses,Median_Loss_Difference,Subclasses_ARDL_Better,Subclasses_Alternative_Better,Wilcoxon_Statistic,P_Value,Holm_Adjusted_P_Value,Significant_After_Holm
0,ARDL | Symmetric,Persistence | Lag-1 benchmark,46,0.336083,34,12,241.000000,0.000779,0.007008,True
1,ARDL | Symmetric,VAR | Bivariate system,46,-0.142355,12,34,324.000000,0.017270,0.138161,False
2,ARDL | Symmetric,Random Forest | Asymmetric,46,-0.170265,14,32,340.000000,0.027923,0.195458,False
3,ARDL | Symmetric,Random Forest | Symmetric,46,-0.160663,13,33,350.000000,0.037062,0.222373,False
4,ARDL | Symmetric,Ridge | Symmetric,46,-0.127229,14,32,360.000000,0.048571,0.242855,False
5,ARDL | Symmetric,Ridge | Asymmetric,46,-0.122976,14,32,367.000000,0.058261,0.242855,False
6,ARDL | Symmetric,XGBoost | Symmetric,46,-0.139414,14,32,384.000000,0.088419,0.265257,False
7,ARDL | Symmetric,XGBoost | Asymmetric,46,-0.132768,17,29,437.000000,0.263187,0.526373,False
8,ARDL | Symmetric,NARDL | Asymmetric,46,-0.002575,20,26,537.000000,0.974107,0.974107,False


### Interpretation of formal forecast accuracy tests

The symmetric ARDL model recorded the lowest pooled test RMSE. However, its monthly squared-error advantage was not statistically significant against any alternative after applying the Holm correction for multiple comparisons. Some comparisons were significant at the unadjusted 5% level, but these results were not robust to family wise error control.

The monthly tests are based on only 12 test months and consequently have limited statistical power. Failure to reject equal predictive accuracy should therefore not be interpreted as proof that the models have identical forecast performance.

The subclass-level analysis provides a complementary result. ARDL
significantly outperformed the persistence benchmark across the 46 food
subclasses after Holm adjustment. No other comparison remained significant. For most machine learning models, the median subclass loss difference favoured the alternative model even though ARDL achieved the lowest pooled RMSE. This indicates that ARDL's aggregate RMSE advantage was driven by better control of relatively large forecast errors rather than uniform superiority across food subclasses.

ARDL and NARDL produced particularly similar forecast performance. Their small pooled RMSE difference and non-significant monthly and subclass tests provide no robust evidence that asymmetric decomposition improved aggregate forecast accuracy over the symmetric econometric specification.

In [ ]:
# direct symmetric-versus-asymmetric forecast comparisons
def paired_hac_mean_test(loss_difference, max_lag=1):
    """Test whether the mean paired loss difference equals zero."""
    values = np.asarray(loss_difference, dtype=float)
    values = values[np.isfinite(values)]

    observations = len(values)
    mean_difference = values.mean()
    centred_values = values - mean_difference

    long_run_variance = np.dot(
        centred_values,
        centred_values,
    ) / observations

    for lag in range(1, min(max_lag, observations - 1) + 1):
        weight = 1 - lag / (max_lag + 1)
        autocovariance = np.dot(
            centred_values[lag:],
            centred_values[:-lag],
        ) / observations
        long_run_variance += 2 * weight * autocovariance

    long_run_variance = max(long_run_variance, 0.0)
    standard_error = np.sqrt(long_run_variance / observations)

    if np.isclose(standard_error, 0.0):
        test_statistic = 0.0 if np.isclose(mean_difference, 0.0) else np.inf
        p_value = 1.0 if np.isclose(mean_difference, 0.0) else 0.0
    else:
        test_statistic = mean_difference / standard_error

        # Harvey small-sample correction for one-step-ahead forecasts
        correction = np.sqrt((observations - 1) / observations)
        test_statistic *= correction

        p_value = 2 * stats.t.sf(
            np.abs(test_statistic),
            df=observations - 1,
        )

    return {
        "Observations": observations,
        "Mean_Loss_Difference": mean_difference,
        "HAC_Test_Statistic": test_statistic,
        "P_Value": p_value,
    }


representation_pairs = [
    {
        "Comparison": "ARDL versus NARDL",
        "Symmetric_Variant": "ARDL | Symmetric",
        "Asymmetric_Variant": "NARDL | Asymmetric",
    },
    {
        "Comparison": "Ridge",
        "Symmetric_Variant": "Ridge | Symmetric",
        "Asymmetric_Variant": "Ridge | Asymmetric",
    },
    {
        "Comparison": "Random Forest",
        "Symmetric_Variant": "Random Forest | Symmetric",
        "Asymmetric_Variant": "Random Forest | Asymmetric",
    },
    {
        "Comparison": "XGBoost",
        "Symmetric_Variant": "XGBoost | Symmetric",
        "Asymmetric_Variant": "XGBoost | Asymmetric",
    },
]

monthly_variant_loss = (
    forecast_loss_data
    .groupby(
        ["Date", "Forecast_Variant"],
        as_index=False,
    )["Squared_Error"]
    .mean()
)

monthly_representation_records = []

for pair in representation_pairs:
    symmetric_name = pair["Symmetric_Variant"]
    asymmetric_name = pair["Asymmetric_Variant"]

    paired_loss = (
        monthly_variant_loss[
            monthly_variant_loss["Forecast_Variant"].isin(
                [symmetric_name, asymmetric_name]
            )
        ]
        .pivot(
            index="Date",
            columns="Forecast_Variant",
            values="Squared_Error",
        )
        .dropna()
        .sort_index()
    )

    loss_difference = (
        paired_loss[asymmetric_name]
        - paired_loss[symmetric_name]
    )

    test_result = paired_hac_mean_test(
        loss_difference,
        max_lag=1,
    )

    monthly_representation_records.append(
        {
            "Comparison": pair["Comparison"],
            "Symmetric_Variant": symmetric_name,
            "Asymmetric_Variant": asymmetric_name,
            "Symmetric_RMSE": np.sqrt(
                paired_loss[symmetric_name].mean()
            ),
            "Asymmetric_RMSE": np.sqrt(
                paired_loss[asymmetric_name].mean()
            ),
            **test_result,
            "Lower_Loss_Specification": (
                "Symmetric"
                if test_result["Mean_Loss_Difference"] > 0
                else "Asymmetric"
            ),
        }
    )

monthly_representation_tests = pd.DataFrame(
    monthly_representation_records
)

monthly_representation_tests["Holm_Adjusted_P_Value"] = multipletests(
    monthly_representation_tests["P_Value"],
    alpha=0.05,
    method="holm",
)[1]

monthly_representation_tests["Significant_After_Holm"] = (
    monthly_representation_tests["Holm_Adjusted_P_Value"] < 0.05
)

monthly_representation_tests = monthly_representation_tests.sort_values(
    "P_Value"
).reset_index(drop=True)

monthly_representation_tests

,Comparison,Symmetric_Variant,Asymmetric_Variant,Symmetric_RMSE,Asymmetric_RMSE,Observations,Mean_Loss_Difference,HAC_Test_Statistic,P_Value,Lower_Loss_Specification,Holm_Adjusted_P_Value,Significant_After_Holm
0,XGBoost,XGBoost | Symmetric,XGBoost | Asymmetric,1.929075,1.973192,12,0.172156,3.234367,0.007953,Symmetric,0.031813,True
1,Ridge,Ridge | Symmetric,Ridge | Asymmetric,1.984985,1.986590,12,0.006374,2.444139,0.032583,Symmetric,0.097748,False
2,Random Forest,Random Forest | Symmetric,Random Forest | Asymmetric,1.964122,1.991030,12,0.106426,1.485813,0.165416,Symmetric,0.330831,False
3,ARDL versus NARDL,ARDL | Symmetric,NARDL | Asymmetric,1.841097,1.848766,12,0.028298,0.456847,0.656671,Symmetric,0.656671,False


In [ ]:
# subclass-level symmetric versus asymmetric comparisons

subclass_variant_loss = (
    forecast_loss_data
    .groupby(
        [
            "ClassDescription",
            "SubclassDescription",
            "Forecast_Variant",
        ],
        as_index=False,
    )["Squared_Error"]
    .mean()
)

subclass_representation_records = []

for pair in representation_pairs:
    symmetric_name = pair["Symmetric_Variant"]
    asymmetric_name = pair["Asymmetric_Variant"]

    paired_loss = (
        subclass_variant_loss[
            subclass_variant_loss["Forecast_Variant"].isin(
                [symmetric_name, asymmetric_name]
            )
        ]
        .pivot(
            index=["ClassDescription", "SubclassDescription"],
            columns="Forecast_Variant",
            values="Squared_Error",
        )
        .dropna()
    )

    loss_difference = (
        paired_loss[asymmetric_name]
        - paired_loss[symmetric_name]
    )

    if np.allclose(loss_difference, 0.0):
        wilcoxon_statistic = 0.0
        p_value = 1.0
    else:
        wilcoxon_result = stats.wilcoxon(
            loss_difference,
            alternative="two-sided",
            zero_method="wilcox",
        )
        wilcoxon_statistic = float(wilcoxon_result.statistic)
        p_value = float(wilcoxon_result.pvalue)

    subclass_representation_records.append(
        {
            "Comparison": pair["Comparison"],
            "Symmetric_Variant": symmetric_name,
            "Asymmetric_Variant": asymmetric_name,
            "Food_Subclasses": len(loss_difference),
            "Median_Loss_Difference": loss_difference.median(),
            "Subclasses_Symmetric_Better": int(
                (loss_difference > 0).sum()
            ),
            "Subclasses_Asymmetric_Better": int(
                (loss_difference < 0).sum()
            ),
            "Tied_Subclasses": int(
                np.isclose(loss_difference, 0.0).sum()
            ),
            "Wilcoxon_Statistic": wilcoxon_statistic,
            "P_Value": p_value,
        }
    )

subclass_representation_tests = pd.DataFrame(
    subclass_representation_records
)

subclass_representation_tests["Holm_Adjusted_P_Value"] = multipletests(
    subclass_representation_tests["P_Value"],
    alpha=0.05,
    method="holm",
)[1]

subclass_representation_tests["Significant_After_Holm"] = (
    subclass_representation_tests["Holm_Adjusted_P_Value"] < 0.05
)

subclass_representation_tests = subclass_representation_tests.sort_values(
    "P_Value"
).reset_index(drop=True)

subclass_representation_tests

,Comparison,Symmetric_Variant,Asymmetric_Variant,Food_Subclasses,Median_Loss_Difference,Subclasses_Symmetric_Better,Subclasses_Asymmetric_Better,Tied_Subclasses,Wilcoxon_Statistic,P_Value,Holm_Adjusted_P_Value,Significant_After_Holm
0,Ridge,Ridge | Symmetric,Ridge | Asymmetric,46,0.005703,35,11,0,223.000000,0.000341,0.001363,True
1,XGBoost,XGBoost | Symmetric,XGBoost | Asymmetric,46,0.041374,30,16,0,304.000000,0.009015,0.027044,True
2,Random Forest,Random Forest | Symmetric,Random Forest | Asymmetric,46,-0.000133,23,23,0,481.000000,0.522711,1.000000,False
3,ARDL versus NARDL,ARDL | Symmetric,NARDL | Asymmetric,46,-0.002575,20,26,0,537.000000,0.974107,1.000000,False


### Interpretation of Symmetric Asymmetric Comparisons

The direct comparisons provide evidence that the forecasting value of
asymmetric exchange-rate predictors depends on the model.

For XGBoost, the symmetric specification achieved a lower test RMSE than the asymmetric specification. This difference was significant in the monthly HAC test after Holm correction (adjusted p-value = 0.0318). The subclass analysis supports the same conclusion: symmetric XGBoost performed better in 30 of the 46 food subclasses, with a Holm-adjusted p-value of 0.0270.

Symmetric Ridge performed better in 35 of the 46 subclasses. This subclass-level difference remained significant after Holm correction (adjusted p-value = 0.0014). However, its monthly test was not significant after correction, indicating that the size of the aggregate monthly advantage was relatively small.

Random Forest produced an evenly divided result, with each representation performing better in 23 subclasses. Neither the monthly nor subclass-level test identified a statistically significant difference between the symmetric and asymmetric specifications.

ARDL and NARDL also produced statistically indistinguishable forecast
performance. NARDL performed better in 26 subclasses, while ARDL performed better in 20, but the median loss difference was extremely small and the Wilcoxon test was not significant. The slightly lower pooled RMSE of ARDL therefore does not constitute robust evidence that it forecasts better than NARDL.

Overall, asymmetric predictors do not provide a general forecasting advantage. They improve predictions for some food subclasses, but symmetric representations are more effective for XGBoost and Ridge when performance is considered across the complete test sample. The results support treating predictive asymmetry as category- and model-specific rather than universal. Because subclasses may share common price shocks, the subclass Wilcoxon results are interpreted as robustness evidence rather than completely independent cross-sectional tests.

In [ ]:
# plot overall locked-test forecast performance

evaluation_figure_directory = Path(
    "../reports/figures/model_evaluation"
)
evaluation_figure_directory.mkdir(parents=True, exist_ok=True)

performance_plot_data = all_test_model_results.copy()

performance_plot_data["Forecast_Variant"] = (
    performance_plot_data["Model"]
    + " — "
    + performance_plot_data["Representation"]
)

performance_plot_data = performance_plot_data.sort_values(
    "RMSE"
).reset_index(drop=True)

approach_colours = {
    "Econometric": "#1f77b4",
    "Machine learning": "#2ca02c",
    "Benchmark": "#7f7f7f",
}

bar_colours = performance_plot_data["Approach"].map(
    approach_colours
)

figure, axes = plt.subplots(
    ncols=2,
    figsize=(15, 8),
    sharey=True,
)

metrics = [
    ("RMSE", "Root mean squared error"),
    ("MAE", "Mean absolute error"),
]

for axis, (metric, axis_label) in zip(axes, metrics):
    bars = axis.barh(
        performance_plot_data["Forecast_Variant"],
        performance_plot_data[metric],
        color=bar_colours,
        alpha=0.9,
    )

    axis.invert_yaxis()
    axis.set_xlabel(axis_label)
    axis.grid(
        axis="x",
        linestyle="--",
        alpha=0.35,
    )

    for bar, value in zip(
        bars,
        performance_plot_data[metric],
    ):
        axis.text(
            value + 0.012,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            fontsize=9,
        )

axes[0].set_ylabel("Forecast variant")

legend_handles = [
    Patch(
        facecolor=colour,
        label=approach,
    )
    for approach, colour in approach_colours.items()
]

figure.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
)

figure.suptitle(
    "Locked-Test Forecast Performance: January–December 2025",
    fontsize=15,
    fontweight="bold",
)

figure.tight_layout(rect=[0, 0.06, 1, 0.95])

overall_performance_figure = (
    evaluation_figure_directory
    / "overall_test_forecast_performance.png"
)

figure.savefig(
    overall_performance_figure,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved: {overall_performance_figure}")
print(f"Figure exists: {overall_performance_figure.exists()}")